In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
transients = pd.read_parquet(
    "snia_results.parquet",
    engine="fastparquet",
)

In [ ]:
transients

In [ ]:
transients = (
    transients[["diaObjectId", "tns_name", "ra", "decl"]]
    .rename(columns={"decl": "dec"})
    .dropna()
    .reset_index(drop=True)
    .copy()
)

transients["tns_name"] = transients["tns_name"].astype(str)

print(f"Associating {len(transients)} transients")
display(transients)

In [ ]:
from pathlib import Path
import pandas as pd
from scipy.stats import gamma, uniform
from astro_prost import associate_sample

for i in range(len(transients)):
    one_transient = transients.iloc[[i]].reset_index(drop=True)
    name = str(one_transient.iloc[0]["tns_name"])


    try:
        result = associate_sample(
            one_transient,
            catalogs = ["decals", "panstarrs", "skymapper"],
            name_col="tns_name",
            coord_cols=("ra", "dec"),
            priors={"offset": uniform(loc=0, scale=10)},
            likes={"offset": gamma(a=0.75)},
            n_samples=200,
            parallel=False,
            save=False,
        )

        hosts = pd.concat([hosts, result], ignore_index=True)

        # Replace the checkpoint only after the new file has been written.
        temporary = checkpoint.with_suffix(".tmp")
        hosts.to_pickle(temporary)
        temporary.replace(checkpoint)

        completed.add(name)
        print(f"{i + 1}/{len(transients)} saved: {name}")

    except Exception as exc:
        print(f"{i + 1}/{len(transients)} failed: {name}: {exc}")